# Case study: first SQL questions about the Berlin Airbnb listings

*Session 3, block 1 · Author: course team · Licence: CC-BY-4.0*

**Why.** A housing analyst of the city wants a first picture of short-term rentals in Berlin: how many listings
are there per district, what does a night cost, which listings are reviewed, and how full are the calendars for the
summer? The data come in three tables that are linked by the listing id. SQL answers such questions directly where
the data are, and the same queries later run in the team's PostgreSQL database.

We use **DuckDB**, a database that runs inside the Python process and can query Parquet files directly. Nothing
needs to be installed beyond the course environment, and the SQL is the same as in PostgreSQL for everything we use
here.

The data (Inside Airbnb, Berlin snapshot of 26 June 2026, CC BY 4.0, prepared with
`uv run python case-study/prepare_airbnb.py`):

- `listings`: 12,776 listings with district, room type, guests, price per night, minimum stay, reviews and
  registration status;
- `calendar`: one row per listing and night for the next 365 days (4.7 million rows): is the night still free?
- `reviews_monthly`: number of reviews per listing and month since 2009.

**Tasks of the practice block**

1. First queries: size, keys, filters.
2. Listings and median price per district (GROUP BY, HAVING).
3. Join the listings with their reviews and their availability (JOIN).
4. Listings without any review, and reviews without a known listing (LEFT JOIN, NULL).

Cells marked **TODO** are for you. Solutions are not included; compare your numbers with your neighbours.

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data" / "airbnb"
assert (DATA / "listings.parquet").exists(), "run case-study/prepare_airbnb.py first"

import duckdb

con = duckdb.connect()  # an in-memory database
# views: names for the Parquet files, so that we can write FROM listings
for table in ["listings", "calendar", "reviews_monthly"]:
    con.execute(f"CREATE VIEW {table} AS SELECT * FROM read_parquet('{DATA / (table + '.parquet')}')")


def q(sql: str):
    """Run a query and return the result as a pandas DataFrame."""
    return con.sql(sql).df()


q("SELECT (SELECT COUNT(*) FROM listings) AS listings, (SELECT COUNT(*) FROM calendar) AS calendar_rows, "
  "(SELECT COUNT(*) FROM reviews_monthly) AS review_months")

## 1. Look at the tables

`DESCRIBE` lists the columns and their types. A **primary key** identifies each row: `id` in `listings`; in
`calendar` the pair (`listing_id`, `date`), because each listing appears once per night; in `reviews_monthly` the
pair (`listing_id`, `month`). `calendar.listing_id` and `reviews_monthly.listing_id` are **foreign keys**: they refer
to a row of `listings`.

In [ ]:
q("DESCRIBE listings").head(15)

In [ ]:
q("DESCRIBE calendar")

In [ ]:
# is id really unique? Is (listing_id, date) unique in the calendar? A primary key must be.
q("""
SELECT (SELECT COUNT(*) FROM listings) AS n_listings,
       (SELECT COUNT(DISTINCT id) FROM listings) AS n_ids,
       (SELECT COUNT(*) FROM calendar) AS n_calendar_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, date FROM calendar)) AS n_calendar_keys
""")

### First queries: SELECT, WHERE, ORDER BY, LIMIT

A family of four looks for an entire flat in Friedrichshain-Kreuzberg for a weekend: short stay, at least four
guests, at most €150 a night, and at least 20 reviews. We show the id and the numbers, not the title: hosts
sometimes write their own name into the title, and the analysis does not need it.

In [ ]:
q("""
SELECT id, neighbourhood, accommodates, price, number_of_reviews, review_scores_rating
FROM listings
WHERE district = 'Friedrichshain-Kreuzberg'
  AND room_type = 'Entire home/apt'
  AND minimum_nights <= 2 AND accommodates >= 4 AND price <= 150 AND number_of_reviews >= 20
ORDER BY review_scores_rating DESC, number_of_reviews DESC
LIMIT 5
""")

In [ ]:
# text patterns: how many titles mention a balcony (English or German)? ILIKE ignores case
q("SELECT COUNT(*) AS n FROM listings WHERE name ILIKE '%balcony%' OR name ILIKE '%balkon%'")

**TODO 1.** How many listings have no price (`price IS NULL`)? Why does `WHERE price = NULL` return nothing? How
many listings require a minimum stay of 28 nights or more?

In [ ]:
# TODO 1

## 2. Listings and median price per district

`GROUP BY` returns one row per group; `HAVING` filters groups after aggregation. The median price is
`percentile_cont(0.5) WITHIN GROUP (ORDER BY price)` in PostgreSQL and DuckDB (DuckDB also knows `MEDIAN(price)`).
The course compares prices of **short stays** only (`minimum_nights < 28`): a third of the listings require a month
or more, and their price field means something else (Session 4).

In [ ]:
q("""
SELECT district,
       COUNT(*) AS n_listings,
       COUNT(*) FILTER (WHERE minimum_nights < 28 AND price IS NOT NULL) AS n_short_stays,
       percentile_cont(0.5) WITHIN GROUP (ORDER BY price) FILTER (WHERE minimum_nights < 28) AS median_price,
       ROUND(AVG((license_status = 'registration number')::int), 3) AS share_registered
FROM listings
GROUP BY district
HAVING COUNT(*) >= 100
ORDER BY median_price DESC
""")

In [ ]:
# COUNT(*) counts rows, COUNT(column) the non-NULL values, AVG ignores NULLs
q("SELECT COUNT(*) AS n_rows, COUNT(price) AS n_price, ROUND(AVG(price), 1) AS avg_price, "
  "COUNT(bedrooms) AS n_bedrooms FROM listings")

**TODO 2.** Repeat the district table separately for entire homes and private rooms (`GROUP BY district,
room_type`). Keep only groups with at least 30 short stays. Is Mitte the most expensive district for both room
types?

**TODO 3.** For each room type: the number of listings and the share with a registration number. Why do hotel
rooms and shared rooms hardly ever show one?

In [ ]:
# TODO 2 and 3

## 3. Join the listings with their reviews and their availability

An **inner join** returns one row per matching pair. Each review month gets the district and room type of its
listing; each calendar night gets them, too. Aliases (`l`, `r`, `c`) say which table a column comes from.

In [ ]:
# reviews per district and year since 2019 (a proxy for demand)
reviews_by_year = q("""
SELECT l.district, EXTRACT(YEAR FROM r.month)::int AS year, SUM(r.n_reviews) AS n_reviews
FROM listings AS l
JOIN reviews_monthly AS r ON r.listing_id = l.id
WHERE r.month BETWEEN '2019-01-01' AND '2025-12-01'
GROUP BY l.district, year
""")
reviews_by_year.pivot(index="district", columns="year", values="n_reviews").sort_values(2025, ascending=False).head(6)

In [ ]:
# share of nights still free in July and August 2026, by district and room type
q("""
SELECT l.district, l.room_type, COUNT(*) AS nights, ROUND(AVG(c.available::int), 3) AS share_free
FROM listings AS l
JOIN calendar AS c ON c.listing_id = l.id
WHERE c.date BETWEEN '2026-07-01' AND '2026-08-31'
  AND l.room_type IN ('Entire home/apt', 'Private room')
GROUP BY l.district, l.room_type
HAVING COUNT(*) >= 10000
ORDER BY share_free
LIMIT 8
""")

A night that is not available is either **booked or blocked** by the host; the calendar does not say which. A low
share of free nights is therefore an upper bound on demand, not a measure of it. Listings without a single free
night in the whole year are mostly not offered at all.

**Watch the row count.** A join with a one-to-many relationship repeats the rows of the "one" side: after
`listings JOIN reviews_monthly`, a listing with 100 review months appears 100 times. An average of the price over
the joined rows is then weighted by review months:

In [ ]:
q("""
SELECT 'after the join' AS version, COUNT(*) AS n_rows, ROUND(AVG(l.price), 1) AS avg_price
FROM listings AS l JOIN reviews_monthly AS r ON r.listing_id = l.id
WHERE l.minimum_nights < 28
UNION ALL
SELECT 'one row per listing', COUNT(*), ROUND(AVG(price), 1)
FROM listings WHERE minimum_nights < 28 AND number_of_reviews > 0
""")

**TODO 4.** Which ten neighbourhoods (`l.neighbourhood`) had the most reviews in 2025? Add the number of listings
reviewed in 2025 (`COUNT(DISTINCT l.id)`) and the reviews per listing. Does the ranking change?

In [ ]:
# TODO 4

## 4. Listings without any review, and reviews without a known listing

A **left join** keeps every row of the left table; where no partner exists, the columns of the right table are
`NULL`. With `WHERE right.key IS NULL` it finds rows without a partner (an **anti-join**).

In [ ]:
q("""
SELECT l.room_type, COUNT(*) AS n_without_review,
       ROUND(AVG((l.minimum_nights >= 28)::int), 3) AS share_medium_term,
       ROUND(AVG((l.availability_365 = 0)::int), 3) AS share_no_free_night
FROM listings AS l
LEFT JOIN reviews_monthly AS r ON r.listing_id = l.id
WHERE r.listing_id IS NULL
GROUP BY l.room_type
ORDER BY n_without_review DESC
""")

In [ ]:
# for comparison: the same shares among the listings with at least one review
q("""
SELECT COUNT(*) AS n_with_review,
       ROUND(AVG((minimum_nights >= 28)::int), 3) AS share_medium_term,
       ROUND(AVG((availability_365 = 0)::int), 3) AS share_no_free_night
FROM listings WHERE number_of_reviews > 0
""")

Most listings without a review are medium-term rentals: people who rent a flat for three months rarely leave a
review, or the listing is new. A report on "listings that nobody books" based on missing reviews would mostly
describe medium-term rentals.

The other direction: are there review rows or calendar rows whose listing is not in the listings table? The
foreign key would forbid them; files do not.

In [ ]:
q("""
SELECT 'reviews_monthly' AS table_name, COUNT(*) AS rows_without_listing, COUNT(DISTINCT r.listing_id) AS listings
FROM reviews_monthly AS r LEFT JOIN listings AS l ON l.id = r.listing_id
WHERE l.id IS NULL
UNION ALL
SELECT 'calendar', COUNT(*), COUNT(DISTINCT c.listing_id)
FROM calendar AS c LEFT JOIN listings AS l ON l.id = c.listing_id
WHERE l.id IS NULL
""")

57 listings have reviews and 79 have a calendar, but none of them is in the listings table. Inside Airbnb scrapes
the three files separately (the listings between 26 June and 3 July 2026), so a listing that went offline in between
appears in one file only. Every inner join with `listings` drops these rows silently; block 2 makes the database
reject them at load time and records the decision.

**NULL in a left join.** A condition on the right table in `WHERE` turns a left join back into an inner join:

In [ ]:
q("""
SELECT 'condition in WHERE' AS version, COUNT(DISTINCT l.id) AS n_listings
FROM listings AS l LEFT JOIN reviews_monthly AS r ON r.listing_id = l.id
WHERE r.month >= '2026-01-01'
UNION ALL
SELECT 'condition in ON', COUNT(DISTINCT l.id)
FROM listings AS l LEFT JOIN reviews_monthly AS r ON r.listing_id = l.id AND r.month >= '2026-01-01'
""")

**TODO 5.** How many listings have **no review in 2026** (although they may have older reviews)? Write it as a left
join with the condition in `ON` and `WHERE r.listing_id IS NULL`. Which districts have the highest share of such
listings?

**TODO 6.** Weather is not in this notebook's views. Add `weather_daily.parquet` as a view and join it with the
monthly reviews on the month (hint: `date_trunc('month', date)`; average the temperature per month first in a
subquery). Do months with warmer weather have more reviews? Session 12 returns to this question with a proper
model.

In [ ]:
# TODO 5 and 6